In [5]:
# =============================================================================
# 1. УСТАНОВКА НЕОБХОДИМЫХ БИБЛИОТЕК
# =============================================================================

!pip install wfdb scipy scikit-learn matplotlib seaborn pandas imbalanced-learn xgboost lightgbm catboost tensorflow neurokit2 -q

In [6]:
# =============================================================================
# 2. ИМПОРТ БИБЛИОТЕК
# =============================================================================
import wfdb
import numpy as np
import pandas as pd
import zipfile
import os
import shutil
import json
import time

# Обработка сигналов и статистика
from scipy import signal
from scipy.signal import find_peaks, butter, filtfilt, hilbert
from scipy.stats import skew, kurtosis

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns

# Вспомогательные модули
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple
from google.colab import files
from warnings import filterwarnings
from datetime import datetime

# Библиотеки для Baseline A (классический кардио-анализ)
import neurokit2 as nk           # Библиотека для анализа био-сигналов

# Библиотеки для Baseline B (нейросеть)
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau

# Библиотеки для машинного обучения
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from collections import Counter

# Балансировка классов (SMOTE)
from imblearn.over_sampling import SMOTE

# Ансамблевые методы (градиентный бустинг)
from lightgbm import LGBMClassifier
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

# Подавление предупреждений для чистоты вывода
filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

In [7]:
# =============================================================================
# 3. КОНФИГУРАЦИЯ ПАРАМЕТРОВ
# =============================================================================
# Используем dataclass для удобного хранения всех параметров эксперимента

@dataclass
class Config:
    # Параметры фильтрации сигнала (полосовой фильтр)
    lowcut: float = 0.5          # Нижняя частота среза (Гц) - убираем дрейф изолинии
    highcut: float = 45.0        # Верхняя частота среза (Гц) - убираем шумы сети
    filter_order: int = 4        # Порядок фильтра Баттерворта

    # Параметры разделения данных
    test_size: float = 0.3       # 30% данных на тест, 70% на обучение
    random_state: int = 42       # Фиксация seed для воспроизводимости

    # Параметры обработки сигнала
    fs: int = 500                # Частота дискретизации в PTB-XL (500 Гц)
    min_peaks_required: int = 5  # Минимальное количество R-пиков для валидности записи

    # Балансировка классов (SMOTE)
    use_smote: bool = True       # Применять ли аугментацию миноритарных классов

    # Параметры сохранения результатов
    save_results: bool = True
    results_dir: str = None

# Создаём экземпляр конфигурации
config = Config()

# Создаём папку для результатов с временной меткой
if config.save_results:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    config.results_dir = f'./results_{timestamp}'
    os.makedirs(config.results_dir, exist_ok=True)
    print(f"Результаты будут сохранены в: {config.results_dir}")

Результаты будут сохранены в: ./results_20260607_151043


In [8]:
# =============================================================================
# 4. ФУНКЦИЯ ЗАГРУЗКИ ДАННЫХ
# =============================================================================

def upload_and_extract_archive(force_reload: bool = True) -> Optional[str]:
    """
    Загружает архив ptb-xl-balanced.zip через интерфейс Colab и распаковывает его.

    Параметры:
        force_reload: если True, удаляет старые данные перед загрузкой

    Возвращает:
        Путь к папке с данными или None при ошибке
    """
    print("=" * 70)
    print("ЗАГРУЗКА ДАННЫХ PTB-XL")
    print("=" * 70)

    data_path = './data/ptb-xl'

    # Удаляем старые данные, если требуется
    if force_reload and os.path.exists(data_path):
        print("Удаление старых данных...")
        shutil.rmtree(data_path)

    os.makedirs(data_path, exist_ok=True)

    # Проверяем, не загружены ли уже данные
    existing_folders = [d for d in os.listdir(data_path) if d.startswith('record_')]
    if len(existing_folders) > 0 and not force_reload:
        print(f"Данные уже загружены. Найдено {len(existing_folders)} записей.")
        return data_path

    # Инструкция для пользователя
    print("\nИнструкция по загрузке:")
    print("   1. Нажмите на кнопку 'Выбрать файлы' ниже")
    print("   2. Выберите файл ptb-xl-balanced.zip (подготовленный на компьютере)")
    print("   3. Дождитесь завершения загрузки\n")

    # Вызываем виджет загрузки файлов
    uploaded = files.upload()

    if len(uploaded) == 0:
        print("Ошибка: файл не был загружен.")
        return None

    zip_filename = list(uploaded.keys())[0]
    print(f"Загружен архив: {zip_filename}")

    # Очищаем папку назначения
    if os.path.exists(data_path):
        shutil.rmtree(data_path)
    os.makedirs(data_path, exist_ok=True)

    # Распаковка
    print("Распаковка архива...")
    with zipfile.ZipFile(zip_filename, 'r') as zf:
        zf.extractall('.')

    # Автоматически находим распакованную папку
    extracted_folder = None
    for item in os.listdir('.'):
        if os.path.isdir(item):
            for subitem in os.listdir(item):
                if subitem.startswith('record_'):
                    extracted_folder = item
                    break
            if extracted_folder:
                break

    # Перемещаем содержимое в нужное место
    if extracted_folder:
        print(f"  Найдена папка: {extracted_folder}")
        for item in os.listdir(extracted_folder):
            src = os.path.join(extracted_folder, item)
            dst = os.path.join(data_path, item)
            shutil.move(src, dst)
        os.rmdir(extracted_folder)

    # Удаляем ZIP-файл
    os.remove(zip_filename)
    print(f"Архив {zip_filename} удалён")

    # Проверяем результат
    record_folders = [d for d in os.listdir(data_path) if d.startswith('record_')]
    print(f"\nНайдено папок с записями: {len(record_folders)}")

    return data_path

In [9]:
# =============================================================================
# 5. ФУНКЦИЯ ЗАГРУЗКИ МЕТОК (ДИАГНОЗОВ)
# =============================================================================

def load_labels(csv_path: str) -> Dict[int, int]:
    """
    Загружает метки диагнозов из CSV-файла PTB-XL.

    Диагнозы кодируются числами 0-4:
        0 - Норма
        1 - Инфаркт миокарда
        2 - ST-T изменения (STTC)
        3 - Нарушения проводимости
        4 - Гипертрофия

    Параметры:
        csv_path: путь к файлу ptbxl_database.csv

    Возвращает:
        Словарь {ecg_id: diagnosis_label}
    """
    labels_map = {}
    diagnosis_names = {0: 'Норма', 1: 'Инфаркт', 2: 'STTC',
                       3: 'Нарушения проводимости', 4: 'Гипертрофия'}

    if not os.path.exists(csv_path):
        print(f"Ошибка: файл {csv_path} не найден")
        return labels_map

    df = pd.read_csv(csv_path)
    print(f"Загрузка меток из CSV...")

    # Извлекаем метки
    if 'diagnosis' in df.columns:
        for idx, row in df.iterrows():
            ecg_id = row['ecg_id']
            diagnosis = row['diagnosis']
            if diagnosis >= 0:
                labels_map[ecg_id] = int(diagnosis)

        print(f"Загружены метки для {len(labels_map)} записей")

        # Статистика распределения классов
        unique, counts = np.unique(list(labels_map.values()), return_counts=True)
        print(f"Распределение диагнозов:")
        for u, c in zip(unique, counts):
            print(f"    • {diagnosis_names.get(u, u)}: {c} записей ({c/len(labels_map)*100:.1f}%)")

    return labels_map

In [10]:
# =============================================================================
# 6. КЛАСС ОБРАБОТКИ ЭКГ-СИГНАЛОВ (ИЗВЛЕЧЕНИЕ 14 ПРИЗНАКОВ ДЛЯ H1)
# =============================================================================

class ECGProcessor:
    """
    Класс для загрузки, фильтрации и извлечения hand-crafted признаков из ЭКГ-сигнала.

    Извлекаемые признаки:
    1-4: Временные HRV-признаки (SDNN, RMSSD, Mean RR, pNN50)
    5-7: Амплитудные признаки (max, min, размах)
    8-10: Статистические признаки (std, skewness, kurtosis)
    11-12: Спектральные признаки (доминирующая частота, LF/HF)
    13: Энергия сигнала
    14: Количество R-пиков
    """

    def __init__(self, data_path: str):
        """Инициализация с указанием пути к данным PTB-XL."""
        self.data_path = data_path
        self.config = config

    def get_record_folders(self) -> List[str]:
        """Возвращает отсортированный список папок с записями (record_XXX)."""
        folders = [d for d in os.listdir(self.data_path) if d.startswith('record_')]
        return sorted(folders, key=lambda x: int(x.split('_')[1]))

    def load_signal(self, folder_name: str, lead_index: int = 0) -> Tuple[np.ndarray, int, str]:
        """
        Загружает ЭКГ-сигнал из указанной папки и отведения.

        Параметры:
            folder_name: имя папки (например, 'record_100')
            lead_index: индекс отведения (0 для I, 1 для II, и т.д.)

        Возвращает:
            signal_data: массив с отсчётами ЭКГ
            fs: частота дискретизации
            hea_filename: имя файла (без расширения)
        """
        folder_path = os.path.join(self.data_path, folder_name)
        hea_files = [f for f in os.listdir(folder_path) if f.endswith('.hea')]
        if not hea_files:
            raise FileNotFoundError(f"No .hea file in {folder_name}")

        hea_filename = hea_files[0].replace('.hea', '')
        record_path = os.path.join(folder_path, hea_filename)
        record = wfdb.rdrecord(record_path)

        # Если запрошенное отведение отсутствует, берём первое
        if lead_index >= record.p_signal.shape[1]:
            lead_index = 0

        signal_data = record.p_signal[:, lead_index]
        fs = record.fs
        return signal_data, fs, hea_filename

    def preprocess_signal(self, signal_data: np.ndarray, fs: int) -> np.ndarray:
        """
        Предобработка ЭКГ-сигнала:
        1. Полосовая фильтрация 0.5-45 Гц (удаление дрейфа и шумов)
        2. Удаление линейного тренда
        3. Z-нормализация (mean=0, std=1)
        """
        # Полосовой фильтр Баттерворта
        nyquist = fs / 2
        low = self.config.lowcut / nyquist
        high = self.config.highcut / nyquist
        b, a = butter(self.config.filter_order, [low, high], btype='band')
        filtered = filtfilt(b, a, signal_data)

        # Удаление тренда
        filtered = signal.detrend(filtered)

        # Нормализация
        if np.std(filtered) > 0:
            normalized = (filtered - np.mean(filtered)) / np.std(filtered)
        else:
            normalized = filtered - np.mean(filtered)

        return normalized

    def detect_r_peaks(self, ecg_signal: np.ndarray, fs: int) -> np.ndarray:
        """
        Обнаружение R-пиков с использованием огибающей Гильберта.
        Метод: вычисляем аналитический сигнал, находим пики на огибающей.
        """
        # Аналитический сигнал (преобразование Гильберта)
        analytic_signal = hilbert(ecg_signal)
        envelope = np.abs(analytic_signal)

        # Динамический порог - 70-й перцентиль
        threshold = np.percentile(envelope, 70)
        distance = int(0.3 * fs)  # Минимальное расстояние между пиками (300 мс)

        # Поиск пиков на огибающей
        peaks1, _ = find_peaks(envelope, distance=distance, height=threshold)

        # Также ищем пики на инвертированном сигнале (для негативных R-зубцов)
        inverted_signal = -ecg_signal
        envelope_inv = np.abs(hilbert(inverted_signal))
        peaks2, _ = find_peaks(envelope_inv, distance=distance,
                                height=np.percentile(envelope_inv, 70))

        # Объединяем результаты и удаляем дубликаты
        all_peaks = np.unique(np.concatenate([peaks1, peaks2]))

        # Дополнительная фильтрация по амплитуде
        valid_peaks = []
        for peak in all_peaks:
            if peak > 0 and peak < len(ecg_signal):
                if np.abs(ecg_signal[peak]) > 0.3:  # Отбрасываем низкоамплитудные шумы
                    valid_peaks.append(peak)
        valid_peaks = np.array(valid_peaks)

        # Ещё одна проверка минимального расстояния
        if len(valid_peaks) > 1:
            filtered_peaks = [valid_peaks[0]]
            for i in range(1, len(valid_peaks)):
                if valid_peaks[i] - filtered_peaks[-1] >= distance:
                    filtered_peaks.append(valid_peaks[i])
            valid_peaks = np.array(filtered_peaks)

        return valid_peaks

    def extract_features(self, ecg_signal: np.ndarray, r_peaks: np.ndarray, fs: int) -> np.ndarray:
        """
        Извлекает 14 hand-crafted признаков из ЭКГ-сигнала.

        Признаки:
        - HRV: SDNN, RMSSD, Mean RR, pNN50
        - Амплитудные: max, min, размах (peak-to-peak)
        - Статистические: std, skewness (асимметрия), kurtosis (эксцесс)
        - Спектральные: доминирующая частота, LF/HF
        - Энергия, количество R-пиков
        """
        features = []

        # === 1-4: HRV-признаки (на основе RR-интервалов) ===
        if len(r_peaks) >= 2:
            rr_intervals = np.diff(r_peaks) / fs  # RR-интервалы в секундах
            features.append(np.std(rr_intervals))        # SDNN
            features.append(np.sqrt(np.mean(np.diff(rr_intervals) ** 2)))  # RMSSD
            features.append(np.mean(rr_intervals))       # Средний RR
            # pNN50: процент последовательных различий > 50 мс
            pNN50 = np.sum(np.abs(np.diff(rr_intervals)) > 0.05) / len(rr_intervals) * 100
            features.append(pNN50)
        else:
            features.extend([0, 0, 0, 0])  # Если не хватает R-пиков

        # === 5-7: Амплитудные признаки ===
        features.append(np.max(ecg_signal))                     # Максимум
        features.append(np.min(ecg_signal))                     # Минимум
        features.append(np.max(ecg_signal) - np.min(ecg_signal))  # Размах

        # === 8-11: Статистические признаки ===
        features.append(np.std(ecg_signal))          # Стандартное отклонение
        features.append(skew(ecg_signal))            # Асимметрия (skewness)
        features.append(kurtosis(ecg_signal))        # Эксцесс (kurtosis)
        features.append(np.sum(ecg_signal ** 2) / len(ecg_signal))  # Энергия

        # === 12-13: Спектральные признаки ===
        freqs, psd = signal.periodogram(ecg_signal, fs=fs)
        if len(psd) > 0:
            # Доминирующая частота (где максимум спектральной плотности)
            features.append(freqs[np.argmax(psd)])

            # LF/HF (Low Frequency / High Frequency)
            # LF: 0.04-0.15 Гц, HF: 0.15-0.4 Гц
            lf_mask = (freqs >= 0.04) & (freqs <= 0.15)
            hf_mask = (freqs > 0.15) & (freqs <= 0.4)
            lf_power = np.sum(psd[lf_mask]) if np.any(lf_mask) else 0
            hf_power = np.sum(psd[hf_mask]) if np.any(hf_mask) else 0
            lf_hf_ratio = lf_power / hf_power if hf_power > 0 else 0
            features.append(lf_hf_ratio)
        else:
            features.extend([0, 0])  # Если ошибка расчёта спектра

        # === 14: Количество R-пиков ===
        features.append(len(r_peaks))

        return np.array(features)

In [11]:
# =============================================================================
# 7. КЛАСС ДЛЯ BASELINE A: ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ ЧЕРЕЗ NEUROKIT2
# =============================================================================
# NeuroKit2 - это библиотека для анализа био-сигналов (ЭКГ, ЭЭГ, ЭМГ, и т.д.)
# Она автоматически выполняет: очистку сигнала, детекцию R-пиков, расчёт HRV.

class BaselineAExtractor:
    """
    Извлекает 6 признаков через NeuroKit2 (классический кардио-анализ).
    Признаки: SDNN, RMSSD, Mean NN, pNN50, максимальная амплитуда, STD.
    """

    @staticmethod
    def extract_features(ecg_signal: np.ndarray, fs: int) -> np.ndarray:
        """
        Возвращает массив из 6 признаков (или нули, если обработка не удалась).
        """
        try:
            # Очистка сигнала от шумов и артефактов
            cleaned = nk.ecg_clean(ecg_signal, sampling_rate=fs, method='biosppy')

            # Детекция R-пиков
            _, rpeaks = nk.ecg_peaks(cleaned, sampling_rate=fs)

            # Расчёт временных HRV-признаков
            hrv_features = nk.hrv_time(rpeaks, sampling_rate=fs, show=False)

            if not hrv_features.empty:
                return np.array([
                    hrv_features.get('HRV_SDNN', [0])[0],     # SDNN
                    hrv_features.get('HRV_RMSSD', [0])[0],   # RMSSD
                    hrv_features.get('HRV_MeanNN', [0])[0],  # Средний RR
                    hrv_features.get('HRV_pNN50', [0])[0],   # pNN50
                    np.max(cleaned),                          # Максимум
                    np.std(cleaned)                           # Стандартное отклонение
                ])
        except Exception as e:
            pass  # В случае ошибки возвращаем нули

        return np.zeros(6)  # Запасной вариант (если запись невалидна)

In [12]:
# =============================================================================
# 8. КЛАСС ДЛЯ BASELINE B: АРХИТЕКТУРА ResNet-1D ДЛЯ ЭКГ
# =============================================================================
# ResNet-1D - глубокая свёрточная нейросеть с остаточными связями.
# Позволяет классифицировать сырые сигналы без ручного извлечения признаков.

class ResNet1D:
    """
    Строит модель свёрточной нейросети (1D) с архитектурой ResNet.
    """

    @staticmethod
    def build_model(input_shape=(5000, 1), num_classes=5):
        """
        Параметры:
            input_shape: форма входного сигнала (длина_сигнала, каналы)
            num_classes: количество классов (5 диагнозов)

        Возвращает:
            Скомпилированная Keras-модель
        """
        inputs = layers.Input(shape=input_shape)

        # Начальный свёрточный блок
        x = layers.Conv1D(64, 5, padding='same', activation='relu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)

        # Остаточный блок 1 (ResNet block)
        shortcut = layers.Conv1D(128, 1, padding='same')(x)  # Проекция для согласования размерностей
        shortcut = layers.BatchNormalization()(shortcut)

        y = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
        y = layers.BatchNormalization()(y)
        y = layers.Conv1D(128, 3, padding='same', activation='relu')(y)
        y = layers.BatchNormalization()(y)
        y = layers.add([y, shortcut])  # Остаточное соединение (skip connection)
        y = layers.Activation('relu')(y)
        y = layers.MaxPooling1D(2)(y)

        # Финальные слои: свёртка + глобальный пулинг + классификация
        y = layers.Conv1D(256, 3, activation='relu')(y)
        y = layers.GlobalAveragePooling1D()(y)
        y = layers.Dropout(0.5)(y)  # Dropout для регуляризации
        outputs = layers.Dense(num_classes, activation='softmax')(y)

        # Компиляция модели
        model = models.Model(inputs, outputs)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='sparse_categorical_crossentropy',  # для целочисленных меток
            metrics=['accuracy']
        )
        return model

In [13]:
# =============================================================================
# 9. КЛАСС ДЛЯ ОТСЛЕЖИВАНИЯ ПРОГРЕССА ОБУЧЕНИЯ НЕЙРОСЕТИ
# =============================================================================
# Пользовательский callback для Keras с детальным выводом в реальном времени.

class DetailedProgressCallback(Callback):
    """
    Выводит прогресс обучения каждой эпохи:
    - время эпохи и общее время
    - accuracy и loss (train и validation)
    - текущий learning rate
    - лучшее значение val_accuracy
    """

    def __init__(self, train_samples=None, verbose=True):
        super().__init__()
        self.verbose = verbose
        self.start_time = None
        self.epoch_start_time = None
        self.best_val_accuracy = 0
        self.best_epoch = 0
        self.train_samples = train_samples
        self.history = {'epoch': [], 'accuracy': [], 'loss': [],
                        'val_accuracy': [], 'val_loss': [], 'time': []}

    def on_train_begin(self, logs={}):
        self.start_time = time.time()
        if self.verbose:
            print(f"\n   {'='*60}")
            print(f"   НАЧАЛО ОБУЧЕНИЯ НЕЙРОСЕТИ (Baseline B)")
            print(f"   {'='*60}")
            if self.train_samples:
                print(f"    Размер обучающей выборки: {self.train_samples} образцов")
            print(f"    Максимальное количество эпох: 20 (с EarlyStopping)\n")

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_start_time = time.time()
        if self.verbose:
            print(f"    Эпоха {epoch+1}/20 ... ", end="", flush=True)

    def on_epoch_end(self, epoch, logs={}):
        epoch_time = time.time() - self.epoch_start_time
        total_time = time.time() - self.start_time
        accuracy = logs.get('accuracy', 0)
        loss = logs.get('loss', 0)
        val_accuracy = logs.get('val_accuracy', 0)
        val_loss = logs.get('val_loss', 0)

        # Сохраняем историю
        self.history['epoch'].append(epoch + 1)
        self.history['accuracy'].append(accuracy)
        self.history['loss'].append(loss)
        self.history['val_accuracy'].append(val_accuracy)
        self.history['val_loss'].append(val_loss)
        self.history['time'].append(epoch_time)

        # Получаем текущий learning rate
        try:
            lr = self.model.optimizer.learning_rate.numpy()
        except:
            try:
                lr = self.model.optimizer.lr.numpy()
            except:
                lr = 0.001

        # Обновляем лучший результат
        if val_accuracy > self.best_val_accuracy:
            self.best_val_accuracy = val_accuracy
            self.best_epoch = epoch + 1

        if self.verbose:
            print(f"за {epoch_time:.1f} сек")
            print(f"       accuracy: {accuracy:.4f} | loss: {loss:.4f}")
            print(f"       val_accuracy: {val_accuracy:.4f} | val_loss: {val_loss:.4f}")
            print(f"       Всего прошло: {total_time/60:.1f} мин | lr: {lr:.6f}")
            print(f"       Лучшая val_accuracy: {self.best_val_accuracy:.4f} (эпоха {self.best_epoch})\n")

    def on_train_end(self, logs={}):
        total_time = time.time() - self.start_time
        if self.verbose:
            print(f"   {'='*60}")
            print(f"   ОБУЧЕНИЕ ЗАВЕРШЕНО")
            print(f"   {'='*60}")
            print(f"    Общее время обучения: {total_time/60:.1f} минут")
            print(f"    Лучшая val_accuracy: {self.best_val_accuracy:.4f} (эпоха {self.best_epoch})")
            print(f"   {'='*60}\n")

In [14]:
# =============================================================================
# 10. ФУНКЦИЯ СОХРАНЕНИЯ РЕЗУЛЬТАТОВ (JSON, CSV, PNG)
# =============================================================================

def save_all_results(results_dict, config):
    """
    Сохраняет все результаты эксперимента в папку results_{timestamp}:
    - summary_metrics.json: основные метрики (F1, accuracy)
    - table_all_methods.csv: сравнение всех моделей ML
    - confusion_matrix.csv: матрица ошибок H1
    - feature_importance.csv: важность признаков
    - calibration_*.csv: результаты калибровки гиперпараметров
    - neural_network_training_history.csv: история обучения нейросети
    - comparison_bar_chart.png: столбчатая диаграмма сравнения методов
    - confusion_matrix.png: визуализация матрицы ошибок
    - feature_importance.png: диаграмма важности признаков
    """
    if not config.save_results:
        return

    results_dir = config.results_dir
    print(f"\nСохранение результатов в {results_dir}...")

    # 1. Сохраняем основные метрики в JSON
    summary = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'H1_f1': float(results_dict['H1_f1']),
        'Baseline_A_f1': float(results_dict['Baseline_A_f1']),
        'Baseline_B_f1': float(results_dict['Baseline_B_f1']),
        'H1_accuracy': float(results_dict['H1_accuracy']),
        'Baseline_A_accuracy': float(results_dict['Baseline_A_accuracy']),
        'Baseline_B_accuracy': float(results_dict['Baseline_B_accuracy'])
    }

    with open(os.path.join(results_dir, 'summary_metrics.json'), 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    # 2. Сохраняем таблицу всех методов ML
    if 'table2_results' in results_dict and results_dict['table2_results']:
        df_table2 = pd.DataFrame(results_dict['table2_results']).T
        df_table2.to_csv(os.path.join(results_dir, 'table_all_methods.csv'), encoding='utf-8-sig')

    # 3. Сохраняем матрицу ошибок
    if 'confusion_matrix' in results_dict:
        np.savetxt(os.path.join(results_dir, 'confusion_matrix.csv'),
                   results_dict['confusion_matrix'], delimiter=',', fmt='%d')

    # 4. Сохраняем важность признаков
    if 'feature_importance' in results_dict and 'feature_names' in results_dict:
        df_importance = pd.DataFrame({
            'feature': results_dict['feature_names'],
            'importance': results_dict['feature_importance']
        }).sort_values('importance', ascending=False)
        df_importance.to_csv(os.path.join(results_dir, 'feature_importance.csv'),
                            index=False, encoding='utf-8-sig')

    # 5. Сохраняем результаты калибровки (если есть)
    if 'calibration_results' in results_dict:
        for name, df in results_dict['calibration_results'].items():
            if not df.empty:
                df.to_csv(os.path.join(results_dir, f'calibration_{name}.csv'),
                         index=False, encoding='utf-8-sig')

    # 6. Сохраняем историю обучения нейросети
    if 'training_history' in results_dict:
        df_history = pd.DataFrame(results_dict['training_history'])
        df_history.to_csv(os.path.join(results_dir, 'neural_network_training_history.csv'),
                         index=False)

    print(f"Все результаты сохранены в папку: {results_dir}")

In [15]:
# =============================================================================
# 11. ФУНКЦИЯ КАЛИБРОВКИ ГИПЕРПАРАМЕТРОВ
# =============================================================================
# Проверяем влияние параметров моделей через кросс-валидацию (5-fold).

def run_hyperparameter_calibration(X_train, y_train, config):
    """
    Выполняет калибровку гиперпараметров для Random Forest и LightGBM.

    Исследуемые параметры:
    - RF: n_estimators (10-300), max_depth (3-∞)
    - LightGBM: n_estimators (25-300), learning_rate (0.01-0.3)
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.random_state)
    results_calib = {}

    print("\n" + "=" * 80)
    print("КАЛИБРОВКА ГИПЕРПАРАМЕТРОВ")
    print("=" * 80)

    # Эксперимент 1: Random Forest - n_estimators
    print("\nЭксперимент 1: Random Forest - n_estimators")
    n_estimators_range = [10, 25, 50, 75, 100, 150, 200, 250, 300]
    rf_n_results = []

    for n in n_estimators_range:
        start = time.time()
        rf = RandomForestClassifier(n_estimators=n, max_depth=10, min_samples_split=10,
                                     random_state=config.random_state, n_jobs=-1)
        scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
        rf_n_results.append({'n_estimators': n, 'Mean F1': scores.mean(), 'Std F1': scores.std(),
                              'Time': time.time() - start})
        print(f"  n={n:3d} | F1={scores.mean():.4f}±{scores.std():.4f} | Time={rf_n_results[-1]['Time']:.1f}s")

    results_calib['rf_n_estimators'] = pd.DataFrame(rf_n_results)

    # Эксперимент 2: Random Forest - max_depth
    print("\nЭксперимент 2: Random Forest - max_depth")
    depth_range = [3, 5, 8, 10, 12, 15, 20, 25, None]
    rf_depth_results = []

    for depth in depth_range:
        start = time.time()
        rf = RandomForestClassifier(n_estimators=100, max_depth=depth, min_samples_split=10,
                                     random_state=config.random_state, n_jobs=-1)
        scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
        depth_str = '∞' if depth is None else depth
        rf_depth_results.append({'max_depth': depth_str, 'Mean F1': scores.mean(),
                                  'Std F1': scores.std(), 'Time': time.time() - start})
        print(f"  depth={depth_str:>3} | F1={scores.mean():.4f}±{scores.std():.4f} | Time={rf_depth_results[-1]['Time']:.1f}s")

    results_calib['rf_max_depth'] = pd.DataFrame(rf_depth_results)

    # Эксперимент 3: LightGBM - n_estimators
    print("\nЭксперимент 3: LightGBM - n_estimators")
    lgb_n_range = [25, 50, 75, 100, 150, 200, 300]
    lgb_n_results = []

    for n in lgb_n_range:
        start = time.time()
        lgb = LGBMClassifier(n_estimators=n, max_depth=10, learning_rate=0.1,
                              random_state=config.random_state, verbose=-1, n_jobs=-1)
        scores = cross_val_score(lgb, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
        lgb_n_results.append({'n_estimators': n, 'Mean F1': scores.mean(),
                               'Std F1': scores.std(), 'Time': time.time() - start})
        print(f"  n={n:3d} | F1={scores.mean():.4f}±{scores.std():.4f} | Time={lgb_n_results[-1]['Time']:.1f}s")

    results_calib['lgb_n_estimators'] = pd.DataFrame(lgb_n_results)

    # Эксперимент 4: LightGBM - learning_rate
    print("\nЭксперимент 4: LightGBM - learning_rate")
    lr_range = [0.01, 0.03, 0.05, 0.1, 0.2, 0.3]
    lgb_lr_results = []

    for lr in lr_range:
        start = time.time()
        lgb = LGBMClassifier(n_estimators=100, max_depth=10, learning_rate=lr,
                              random_state=config.random_state, verbose=-1, n_jobs=-1)
        scores = cross_val_score(lgb, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
        lgb_lr_results.append({'learning_rate': lr, 'Mean F1': scores.mean(),
                                'Std F1': scores.std(), 'Time': time.time() - start})
        print(f"  lr={lr:.2f} | F1={scores.mean():.4f}±{scores.std():.4f} | Time={lgb_lr_results[-1]['Time']:.1f}s")

    results_calib['lgb_learning_rate'] = pd.DataFrame(lgb_lr_results)

    return results_calib

In [16]:
# =============================================================================
# 12. ОСНОВНАЯ ФУНКЦИЯ: ПОЛНЫЙ ЦИКЛ ЭКСПЕРИМЕНТА
# =============================================================================

def run_ecg_experiment(force_reload: bool = True, max_records: int = None):
    """
    Запускает полный эксперимент сравнения трёх подходов к классификации ЭКГ.

    Параметры:
        force_reload: перезагрузить данные (удалить старые)
        max_records: ограничить количество записей (для теста)

    Возвращает:
        словарь со всеми результатами или None при ошибке
    """

    print("\n" + "=" * 70)
    print("ЭКСПЕРИМЕНТАЛЬНОЕ СРАВНЕНИЕ ПОДХОДОВ К КЛАССИФИКАЦИИ ЭКГ")
    print("=" * 70)
    print("\nУчастники эксперимента:")
    print("   1. Baseline A: NeuroKit2 + Logistic Regression (классический подход)")
    print("   2. Baseline B: ResNet-1D (нейросетевой SOTA подход)")
    print("   3. H1: LightGBM + 14 hand-crafted признаков (предлагаемый метод)")

    # ==================== ЭТАП 1: ЗАГРУЗКА ДАННЫХ ====================
    data_path = upload_and_extract_archive(force_reload=force_reload)
    if data_path is None:
        print("Ошибка: данные не загружены. Эксперимент прерван.")
        return None

    processor = ECGProcessor(data_path)
    csv_path = os.path.join(data_path, 'ptbxl_database.csv')
    labels_map = load_labels(csv_path)

    record_folders = processor.get_record_folders()

    if max_records:
        record_folders = record_folders[:max_records]
        print(f"\nРежим тестирования: обрабатывается только {max_records} записей")

    print(f"\nВсего записей для обработки: {len(record_folders)}")
    print("\n" + "-" * 70)
    print("ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ ДЛЯ ТРЁХ ПОДХОДОВ...")
    print("-" * 70)

    # ==================== ЭТАП 2: ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ ====================
    h1_features = []        # 14 признаков (H1)
    baseline_a_features = [] # 6 признаков (NeuroKit2)
    raw_signals = []         # сырые сигналы для нейросети
    target_labels = []       # метки диагнозов

    for i, folder in enumerate(record_folders):
        try:
            ecg_id = int(folder.split('_')[1])
            if ecg_id not in labels_map:
                continue

            # --- Выбор лучшего отведения (с максимальным количеством R-пиков) ---
            best_lead = 0
            best_peaks = 0

            folder_path_tmp = os.path.join(data_path, folder)
            hea_files_tmp = [f for f in os.listdir(folder_path_tmp) if f.endswith('.hea')]
            if not hea_files_tmp:
                continue

            hea_filename_tmp = hea_files_tmp[0].replace('.hea', '')
            record_path_tmp = os.path.join(folder_path_tmp, hea_filename_tmp)
            record_tmp = wfdb.rdrecord(record_path_tmp)

            # Проверяем первые 12 отведений
            for lead in range(min(record_tmp.p_signal.shape[1], 12)):
                signal_data, fs, _ = processor.load_signal(folder, lead)
                filtered = processor.preprocess_signal(signal_data, fs)
                r_peaks = processor.detect_r_peaks(filtered, fs)
                if len(r_peaks) > best_peaks:
                    best_peaks = len(r_peaks)
                    best_lead = lead

            if best_peaks < config.min_peaks_required:
                continue  # Запись содержит слишком мало R-пиков

            # --- Извлечение признаков с лучшим отведением ---
            signal_data, fs, _ = processor.load_signal(folder, best_lead)
            filtered = processor.preprocess_signal(signal_data, fs)
            r_peaks = processor.detect_r_peaks(filtered, fs)

            features_h1 = processor.extract_features(filtered, r_peaks, fs)
            h1_features.append(features_h1)

            features_a = BaselineAExtractor.extract_features(signal_data, fs)
            baseline_a_features.append(features_a)

            # Приведение всех сигналов к фиксированной длине 5000 отсчётов
            if len(filtered) >= 5000:
                raw_sig = filtered[:5000]
            else:
                raw_sig = np.pad(filtered, (0, 5000 - len(filtered)), 'constant')
            raw_signals.append(raw_sig)

            target_labels.append(labels_map[ecg_id])

            # Прогресс-бар (каждые 200 записей)
            if (i + 1) % 200 == 0 or (i + 1) == len(record_folders):
                print(f"   Обработано {i+1:4d}/{len(record_folders)} записей, "
                      f"успешно: {len(h1_features):4d}")

        except Exception as e:
            continue

    # Конвертируем списки в numpy-массивы
    X_h1_original = np.array(h1_features)
    X_baseline_a_original = np.array(baseline_a_features)
    X_raw_original = np.array(raw_signals)
    y_original = np.array(target_labels)

    print("\n" + "-" * 70)
    print("СТАТИСТИКА ОБРАБОТКИ")
    print("-" * 70)
    print(f"   Успешно обработано записей: {len(X_h1_original)}")
    print(f"   H1 (14 признаков):           {X_h1_original.shape}")
    print(f"   Baseline A (6 признаков):    {X_baseline_a_original.shape}")
    print(f"   Baseline B (сырые сигналы):  {X_raw_original.shape}")

    if len(X_h1_original) < 100:
        print("\nОшибка: недостаточно валидных записей для обучения (<100)")
        return None

    # ==================== ЭТАП 3: БАЛАНСИРОВКА КЛАССОВ (SMOTE) ====================
    print("\n" + "-" * 70)
    print("БАЛАНСИРОВКА КЛАССОВ")
    print("-" * 70)

    if config.use_smote:
        unique, counts = np.unique(y_original, return_counts=True)
        min_class_size = min(counts)
        k_neighbors = min(5, min_class_size - 1)

        if k_neighbors >= 1:
            smote = SMOTE(random_state=config.random_state, k_neighbors=k_neighbors)
            print(f"   Применяем SMOTE (k_neighbors={k_neighbors}) для H1 и Baseline A")

            # SMOTE применяется только к tabular данным, сырые сигналы не аугментируем
            X_h1_balanced, y_h1_balanced = smote.fit_resample(X_h1_original, y_original)
            X_baseline_a_balanced, _ = smote.fit_resample(X_baseline_a_original, y_original)
            X_raw_balanced = X_raw_original
            y_raw_balanced = y_original

            print(f"   После SMOTE (H1, Baseline A): {X_h1_balanced.shape[0]} образцов")
            print(f"   Нейросеть (Baseline B): {X_raw_balanced.shape[0]} образцов (без аугментации)")
        else:
            print(f"   SMOTE не применён (миноритарный класс слишком мал: {min_class_size})")
            X_h1_balanced = X_h1_original
            X_baseline_a_balanced = X_baseline_a_original
            X_raw_balanced = X_raw_original
            y_h1_balanced = y_original
            y_raw_balanced = y_original
    else:
        X_h1_balanced = X_h1_original
        X_baseline_a_balanced = X_baseline_a_original
        X_raw_balanced = X_raw_original
        y_h1_balanced = y_original
        y_raw_balanced = y_original

    # ==================== ЭТАП 4: РАЗДЕЛЕНИЕ НА ОБУЧАЮЩУЮ И ТЕСТОВУЮ ВЫБОРКИ ====================
    print("\n" + "-" * 70)
    print("РАЗДЕЛЕНИЕ ДАННЫХ (обучающая: 70%, тестовая: 30%)")
    print("-" * 70)

    X_train_h1, X_test_h1, y_train_h1, y_test_h1 = train_test_split(
        X_h1_balanced, y_h1_balanced,
        test_size=config.test_size,
        random_state=config.random_state,
        stratify=y_h1_balanced
    )

    X_train_a, X_test_a, _, _ = train_test_split(
        X_baseline_a_balanced, y_h1_balanced,
        test_size=config.test_size,
        random_state=config.random_state,
        stratify=y_h1_balanced
    )

    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X_raw_balanced, y_raw_balanced,
        test_size=config.test_size,
        random_state=config.random_state,
        stratify=y_raw_balanced
    )

    # StandardScaler для tabular данных (H1 и Baseline A)
    scaler = StandardScaler()
    X_train_h1_scaled = scaler.fit_transform(X_train_h1)
    X_test_h1_scaled = scaler.transform(X_test_h1)

    print(f"   H1 / Baseline A: обучающая={len(X_train_h1)}, тестовая={len(X_test_h1)}")
    print(f"   Нейросеть:        обучающая={len(X_train_raw)}, тестовая={len(X_test_raw)}")

    # ==================== КАЛИБРОВКА ГИПЕРПАРАМЕТРОВ ====================
    calibration_results = run_hyperparameter_calibration(X_train_h1, y_train_h1, config)

    # ==================== ЭТАП 5: BASELINE A (NEUROKIT2 + ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ) ====================
    print("\n" + "=" * 70)
    print("BASELINE A: NeuroKit2 + Логистическая регрессия")
    print("=" * 70)

    X_train_a_scaled = scaler.fit_transform(X_train_a)
    X_test_a_scaled = scaler.transform(X_test_a)

    lr = LogisticRegression(max_iter=1000, random_state=config.random_state, penalty='l2', C=1.0)
    lr.fit(X_train_a_scaled, y_train_h1)
    y_pred_a = lr.predict(X_test_a_scaled)

    f1_a = f1_score(y_test_h1, y_pred_a, average='weighted')
    acc_a = accuracy_score(y_test_h1, y_pred_a)

    print(f"   Weighted F1-score: {f1_a:.4f}")
    print(f"   Accuracy:          {acc_a:.4f}")

    # ==================== ЭТАП 6: BASELINE B (ResNet-1D) ====================
    print("\n" + "=" * 70)
    print("BASELINE B: ResNet-1D (Сверточная нейросеть)")
    print("=" * 70)

    # Изменяем форму: (образцы, длина_сигнала) -> (образцы, длина_сигнала, 1 канал)
    X_train_r = X_train_raw.reshape((X_train_raw.shape[0], X_train_raw.shape[1], 1))
    X_test_r = X_test_raw.reshape((X_test_raw.shape[0], X_test_raw.shape[1], 1))

    print(f"    Размер обучающей выборки: {X_train_r.shape[0]} сигналов")
    print(f"    Размер тестовой выборки: {X_test_r.shape[0]} сигналов")
    print(f"    Формат одного сигнала: {X_train_r.shape[1]} отсчётов × 1 канал")

    model = ResNet1D.build_model(input_shape=(X_train_r.shape[1], 1), num_classes=5)

    # Callbacks для управления обучением
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=0)
    detailed_progress = DetailedProgressCallback(train_samples=X_train_r.shape[0], verbose=True)

    print("\n   ЗАПУСК ОБУЧЕНИЯ...")

    history = model.fit(
        X_train_r, y_train_raw,
        epochs=20,
        batch_size=64,
        validation_split=0.1,
        callbacks=[early_stop, reduce_lr, detailed_progress],
        verbose=0
    )

    print("    Выполнение предсказаний на тестовой выборке...")
    y_pred_b = np.argmax(model.predict(X_test_r, verbose=0), axis=1)

    f1_b = f1_score(y_test_raw, y_pred_b, average='weighted')
    acc_b = accuracy_score(y_test_raw, y_pred_b)

    print(f"\n   РЕЗУЛЬТАТЫ BASELINE B:")
    print(f"   Weighted F1-score: {f1_b:.4f}")
    print(f"   Accuracy:          {acc_b:.4f}")

    # ==================== ЭТАП 7: ПРЕДЛАГАЕМЫЙ МЕТОД H1 (LIGHTGBM + 14 ПРИЗНАКОВ) ====================
    print("\n" + "=" * 70)
    print("ПРЕДЛАГАЕМЫЙ МЕТОД H1: LightGBM + 14 hand-crafted признаков")
    print("=" * 70)

    model_h1 = LGBMClassifier(
        n_estimators=100,
        max_depth=10,
        learning_rate=0.1,
        random_state=config.random_state,
        verbose=-1
    )
    model_h1.fit(X_train_h1, y_train_h1)
    y_pred_h1 = model_h1.predict(X_test_h1)

    f1_h1 = f1_score(y_test_h1, y_pred_h1, average='weighted')
    acc_h1 = accuracy_score(y_test_h1, y_pred_h1)

    print(f"   Weighted F1-score: {f1_h1:.4f}")
    print(f"   Accuracy:          {acc_h1:.4f}")

    # ==================== ЭТАП 8: ТАБЛИЦА ВСЕХ МЕТОДОВ ML ====================
    print("\n" + "=" * 70)
    print("ТАБЛИЦА. Оценка всех методов машинного обучения")
    print("=" * 70)
    print(f"{'Модель':<25} | {'Weighted F1':<12} | {'Accuracy':<12}")
    print("-" * 70)

    models_table2 = {
        'logistic_regression': LogisticRegression(max_iter=1000, random_state=config.random_state),
        'knn': KNeighborsClassifier(n_neighbors=5),
        'decision_tree': DecisionTreeClassifier(max_depth=10, random_state=config.random_state),
        'naive_bayes': GaussianNB(),
        'lda': LinearDiscriminantAnalysis(),
        'svm': SVC(kernel='rbf', random_state=config.random_state),
        'random_forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=config.random_state),
        'extra_trees': ExtraTreesClassifier(n_estimators=100, max_depth=10, random_state=config.random_state),
        'adaboost': AdaBoostClassifier(n_estimators=50, random_state=config.random_state),
        'gradient_boosting': GradientBoostingClassifier(n_estimators=100, random_state=config.random_state),
        'xgboost': XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='mlogloss',
                                 verbosity=0, random_state=config.random_state) if XGB_AVAILABLE else None,
        'lightgbm': LGBMClassifier(n_estimators=100, random_state=config.random_state, verbose=-1),
        'catboost': CatBoostClassifier(iterations=100, depth=6, verbose=False,
                                       random_seed=config.random_state) if CATBOOST_AVAILABLE else None,
        'mlp': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200, random_state=config.random_state)
    }

    table2_results = {}

    for name, model in models_table2.items():
        if model is None:
            print(f"{name:25} | {'N/A':>10}   | {'N/A':>10}")
            continue
        try:
            model.fit(X_train_h1_scaled, y_train_h1)
            y_pred = model.predict(X_test_h1_scaled)
            f1 = f1_score(y_test_h1, y_pred, average='weighted')
            acc = accuracy_score(y_test_h1, y_pred)
            table2_results[name] = {'f1': f1, 'acc': acc}
            print(f"{name:25} | {f1:>10.4f}   | {acc:>10.4f}")
        except Exception as e:
            print(f"{name:25} | Ошибка: {str(e)[:30]}")

    # ==================== ЭТАП 9: ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ ====================
    print("\n" + "=" * 70)
    print("ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
    print("=" * 70)

    # График 1: Сравнение трёх подходов (столбчатая диаграмма)
    plt.figure(figsize=(10, 6))
    methods = ['Baseline A\n(NeuroKit2 + LR)', 'Baseline B\n(ResNet-1D)', 'H1\n(LightGBM + 14 пр.)']
    f1_scores = [f1_a, f1_b, f1_h1]
    acc_scores = [acc_a, acc_b, acc_h1]

    x = np.arange(len(methods))
    width = 0.35

    bars1 = plt.bar(x - width/2, f1_scores, width, label='Weighted F1-score', color='steelblue')
    bars2 = plt.bar(x + width/2, acc_scores, width, label='Accuracy', color='coral')

    for bar in bars1:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

    plt.ylabel('Значение метрики', fontsize=12)
    plt.title('Сравнение подходов к классификации ЭКГ', fontsize=14)
    plt.xticks(x, methods, fontsize=10)
    plt.legend(loc='upper right')
    plt.ylim(0, 0.55)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    if config.save_results:
        plt.savefig(os.path.join(config.results_dir, 'comparison_bar_chart.png'), dpi=300, bbox_inches='tight')
    plt.show()

    # График 2: Матрица ошибок для H1
    class_names = ['Норма', 'Инфаркт', 'STTC', 'Нарушения\nпроводимости', 'Гипертрофия']
    cm = confusion_matrix(y_test_h1, y_pred_h1)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Предсказанный класс', fontsize=12)
    plt.ylabel('Истинный класс', fontsize=12)
    plt.title('Матрица ошибок классификации (H1)', fontsize=14)
    plt.tight_layout()

    if config.save_results:
        plt.savefig(os.path.join(config.results_dir, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.show()

    # ==================== АНАЛИЗ ВАЖНОСТИ ПРИЗНАКОВ ====================
    print("\n" + "=" * 70)
    print("АНАЛИЗ ВАЖНОСТИ ПРИЗНАКОВ (H1)")
    print("=" * 70)

    feature_names = [
        'SDNN', 'RMSSD', 'Средний RR', 'pNN50',
        'Максимум', 'Минимум', 'Размах', 'Стд откл',
        'Асимметрия', 'Эксцесс', 'Энергия',
        'Доминирующая частота', 'LF/HF', 'Кол-во пиков'
    ][:X_h1_balanced.shape[1]]

    # Используем Random Forest для оценки важности признаков
    rf = RandomForestClassifier(n_estimators=100, random_state=config.random_state)
    rf.fit(X_h1_balanced, y_h1_balanced)

    indices = np.argsort(rf.feature_importances_)[::-1][:10]

    plt.figure(figsize=(10, 6))
    plt.barh(range(len(indices)), rf.feature_importances_[indices][::-1], color='steelblue')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices[::-1]])
    plt.xlabel('Важность признака', fontsize=12)
    plt.title('Топ-10 наиболее важных признаков', fontsize=14)
    plt.tight_layout()

    if config.save_results:
        plt.savefig(os.path.join(config.results_dir, 'feature_importance.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print("\nТоп-5 наиболее важных признаков:")
    for i, idx in enumerate(indices[:5]):
        print(f"   {i+1}. {feature_names[idx]}: {rf.feature_importances_[idx]:.4f}")

    # ==================== СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ ====================
    print("\n" + "=" * 70)
    print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ ЭКСПЕРИМЕНТА")
    print("=" * 70)
    print(f"\n{'Метод':<45} | {'Weighted F1':<12} | {'Accuracy':<12}")
    print("-" * 70)
    print(f"{'Baseline A (NeuroKit2 + LR)':<45} | {f1_a:>10.4f}   | {acc_a:>10.4f}")
    print(f"{'Baseline B (ResNet-1D)':<45} | {f1_b:>10.4f}   | {acc_b:>10.4f}")
    print(f"{'H1 (LightGBM + 14 признаков)':<45} | {f1_h1:>10.4f}   | {acc_h1:>10.4f}")

    # Сбор всех результатов в словарь для сохранения
    all_results = {
        'H1_f1': f1_h1,
        'H1_accuracy': acc_h1,
        'Baseline_A_f1': f1_a,
        'Baseline_A_accuracy': acc_a,
        'Baseline_B_f1': f1_b,
        'Baseline_B_accuracy': acc_b,
        'table2_results': table2_results,
        'confusion_matrix': cm,
        'feature_importance': rf.feature_importances_,
        'feature_names': feature_names,
        'calibration_results': calibration_results,
        'training_history': detailed_progress.history
    }

    # Сохранение всех результатов
    save_all_results(all_results, config)

    return all_results

In [ ]:
# =============================================================================
# 13. ЗАПУСК ЭКСПЕРИМЕНТА
# =============================================================================

if __name__ == "__main__":
    results = run_ecg_experiment(force_reload=True, max_records=None)

    if results:
        print("\n" + "=" * 70)
        print("ЭКСПЕРИМЕНТ ЗАВЕРШЁН")
        print("=" * 70)
        print(f"Итоговый Weighted F1-score H1: {results['H1_f1']:.4f}")
        print(f"Baseline A F1: {results['Baseline_A_f1']:.4f}")
        print(f"Baseline B F1: {results['Baseline_B_f1']:.4f}")